In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

results_dir = Path("..") / "ExperimentRunner" / "experiment_results"

supported = {".csv", ".json", ".jsonl", ".parquet"}
files = sorted(
    p for p in results_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in supported
)

def load_experiment_file(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".json":
        data = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(data, list):
            return pd.json_normalize(data)
        if isinstance(data, dict):
            return pd.json_normalize(data)
        return pd.DataFrame({"value": [data]})
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    raise ValueError(f"Unsupported file type: {path}")

csv_path = next((p for p in files if p.suffix.lower() == ".csv"), None)
if csv_path is None:
    raise FileNotFoundError("No .csv experiment file found in 'files'.")

csv_df = load_experiment_file(csv_path).copy()
csv_df = csv_df.sort_values(["Instance", "Evaluations"])

target_instance = "tai_20_10_0"
instance_df = csv_df[csv_df["Instance"] == target_instance].copy()
if instance_df.empty:
    raise ValueError(f"Instance '{target_instance}' not found in {csv_path.name}.")

best_cost = instance_df["BestCost"].min()
instance_df["RelativeBestCostPct"] = ((instance_df["BestCost"] / best_cost) - 1.0) * 100.0

ax = instance_df.plot(
    x="Evaluations",
    y="RelativeBestCostPct",
    marker="o",
    figsize=(10, 6),
    logx=True,
    legend=False,
    color="#1f77b4",
)

ax.set_xlabel("Sample size")
ax.set_ylabel("Best solution relative to best found [%]")
ax.set_title(f"Relative best solution vs sample size ({target_instance})")
ax.grid(True, which="both", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()